## Methodological guardrails

1. A high score is not proof that an intersection is inherently dangerous.
2. Current predictions are not exposure-normalized because pedestrian and vehicle volumes are not yet directly included.
3. Current-week crash history and current-week weather are used to predict the following week.
4. Training, validation, and test periods are chronological.
5. Model selection, threshold tuning, and ablation comparisons use the validation period.
6. The test period is evaluated only after the baseline model and threshold are locked.
7. Feature importance does not establish causation.
8. Engineering recommendations require external road-safety evidence and site review.

**This is Notebook 3** — calibration, feature importance, error analysis, subgroup/time-stability checks, mapping, and recommendation output. It starts by loading the model and test predictions Notebook 2 already saved to disk, so it can be reopened and run on its own without re-running Notebook 1 or Notebook 2.

# 1. Install packages

In [ ]:
# Install (quietly) the libraries this notebook depends on: data handling (pandas/numpy),
# plotting (matplotlib), street-network tools (osmnx/networkx), fast columnar I/O (pyarrow),
# modeling (scikit-learn), model persistence (joblib), and interactive maps (folium).
%pip install -q pandas numpy matplotlib osmnx networkx pyarrow scikit-learn joblib folium xgboost lightgbm

# 2. Imports and reproducibility settings

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path
from typing import Any

import folium
import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# XGBoost gives a gradient-boosted-tree option to compare against logistic
# regression and random forest; it's imported separately since it isn't
# part of scikit-learn itself.
from xgboost import XGBClassifier

# LightGBM is a second, independently-implemented gradient-boosting library —
# comparing it against XGBoost checks whether "gradient boosting works well
# here" is a robust finding, or just an artifact of one specific library.
from lightgbm import LGBMClassifier

# MLPClassifier is a small feed-forward neural network, included as a model
# family that works completely differently from every tree-based model above
# (no splits/thresholds — it learns smooth, weighted combinations of inputs).
from sklearn.neural_network import MLPClassifier

# FutureWarning noise from library version churn just clutters notebook output,
# so it's suppressed here (this doesn't hide real errors, only future-deprecation notices).
warnings.filterwarnings("ignore", category=FutureWarning)

# A fixed random seed makes anything involving randomness (model training,
# sampling, shuffling) reproducible from run to run.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Record the exact library versions used, so results here can be reproduced
# later even if the environment/dependencies change.
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("OSMnx:", ox.__version__)
print("scikit-learn:", sklearn.__version__)

# 3. Configuration

In [ ]:
# Sets thi to a repository folder only when the project is not stored in /content.
# On Colab, leave this as None so choose_repo_root() auto-detects /content.
REPO_ROOT_OVERRIDE = None

# Optional exact paths. Leave as None to use the filename patterns below.
EVENT_DATA_PATH_OVERRIDE = None
OSM_GRAPH_PATH_OVERRIDE = None
WEATHER_DATA_PATH_OVERRIDE = None

# Optional date limits such as "2024-01-01" and "2024-12-31".
# When left as None, the continuous weather file defines the analysis coverage.
ANALYSIS_START_DATE = None
ANALYSIS_END_DATE = None

CANDIDATE_MIN_STREET_COUNT = 3
MAX_NODE_MATCH_DISTANCE_M = 75.0

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

THRESHOLD_BETA = 2.0
RF_N_ESTIMATORS = 200
PERMUTATION_SAMPLE_ROWS = 10_000
PERMUTATION_REPEATS = 5

assert np.isclose(TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION, 1.0)

print("Minimum street count:", CANDIDATE_MIN_STREET_COUNT)
print("Maximum crash-to-node distance:", MAX_NODE_MATCH_DISTANCE_M, "m")

# 4. Project paths

In [ ]:
# Notebook 3 only needs the model/reports folders that Notebook 2 already
# wrote to — it never looks for the raw Notebook 1 files (event/OSM/weather),
# since it doesn't touch that data at all.
def choose_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        return Path(REPO_ROOT_OVERRIDE).expanduser().resolve()

    colab_root = Path("/content")
    current = Path.cwd()
    if (colab_root / "data").exists() or current == Path("/"):
        return colab_root

    if current.name.lower() == "notebooks":
        return current.parent

    return current


REPO_ROOT = choose_repo_root()

DATA_PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "models"
METRIC_DIR = REPO_ROOT / "reports" / "metrics"
NOTEBOOK_OUTPUT_DIR = REPO_ROOT / "reports" / "notebook_02_03"

for directory in [DATA_PROCESSED_DIR, MODEL_DIR, METRIC_DIR, NOTEBOOK_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)

# 5. Load the trained model and test predictions

In [ ]:
# --- Load everything Notebook 2 saved, instead of retraining ---
MODEL_PATH = MODEL_DIR / "safecross_baseline_pipeline.joblib"
MODEL_METADATA_PATH = MODEL_DIR / "safecross_baseline_metadata.json"
TEST_PREDICTION_PATH = DATA_PROCESSED_DIR / "safecross_test_predictions.parquet"
TEST_METRICS_PATH = METRIC_DIR / "selected_model_test_metrics.json"

for required_path in [MODEL_PATH, MODEL_METADATA_PATH, TEST_PREDICTION_PATH, TEST_METRICS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required Notebook 2 output: {required_path}\n\n"
            "Run Notebook 2 completely first, or copy its saved model/reports\n"
            "folders into this project's data/models/reports directories."
        )

# The fitted preprocessing + model pipeline — no retraining needed.
selected_model = joblib.load(MODEL_PATH)

with open(MODEL_METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

# Recover everything downstream cells expect to already be in scope, from
# the metadata Notebook 2 recorded rather than recomputing it.
selected_model_name = metadata["selected_model"]
selected_threshold = float(metadata["selected_threshold"])
TARGET_COLUMN = metadata["target"]
NUMERIC_FEATURES = metadata["numeric_features"]
CATEGORICAL_FEATURES = metadata["categorical_features"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Row-level test data and predictions, saved by Notebook 2. test_data is kept
# as the name used throughout the cells below, so nothing else needs editing.
test_data = pd.read_parquet(TEST_PREDICTION_PATH)

X_test = test_data[ALL_FEATURES]
y_test = test_data[TARGET_COLUMN]
test_probability = test_data["predicted_probability"].to_numpy()
test_prediction = test_data["predicted_class"].to_numpy()

# The already-computed test-set metrics (precision, recall, F2, etc.) from
# Notebook 2 — used by the model card below.
with open(TEST_METRICS_PATH, "r", encoding="utf-8") as file:
    test_metrics = json.load(file)

print("Loaded model:", selected_model_name)
print("Selected threshold:", round(selected_threshold, 4))
print("Test rows loaded:", f"{len(test_data):,}")

# 6. Re-declare evaluation utilities

In [ ]:
# classification_metrics / safe_probability_metric / top_k_hotspot_recovery /
# find_best_fbeta_threshold are needed again here (subgroup and weekly
# performance below call classification_metrics directly). Redefined here so
# this notebook is self-contained rather than depending on Notebook 2's
# in-memory state.
# Wraps a scikit-learn probability-based metric (like ROC-AUC) so that if it
# can't be computed (e.g. only one class present in y_true), the function
# returns NaN instead of crashing the whole notebook.
def safe_probability_metric(metric_function, y_true, probability) -> float:
    try:
        return float(metric_function(y_true, probability))
    except ValueError:
        return np.nan


# Computes a broad set of classification metrics at a given probability
# threshold, bundled into one dictionary for easy comparison across models.
def classification_metrics(y_true, probability, threshold: float) -> dict[str, Any]:
    # Convert continuous predicted probabilities into hard 0/1 predictions by
    # comparing against the chosen threshold.
    prediction = (np.asarray(probability) >= threshold).astype(int)

    # Break the confusion matrix into its four named quadrants for easy reuse below.
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()

    # Fraction of rows that are actually positive — used as a naive baseline for
    # Average Precision (a model no better than guessing the base rate scores ~this).
    positive_rate = float(np.mean(y_true))

    average_precision = safe_probability_metric(average_precision_score, y_true, probability)

    # "Lift" = how many times better than the naive base-rate baseline the model's
    # Average Precision is. A lift of 1 means no better than guessing; higher is better.
    average_precision_lift = average_precision / positive_rate if positive_rate > 0 and np.isfinite(average_precision) else np.nan

    return {
        "threshold": float(threshold),
        "positive_rate": positive_rate,
        "precision": float(precision_score(y_true, prediction, zero_division=0)),
        "recall": float(recall_score(y_true, prediction, zero_division=0)),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
        "f2": float(fbeta_score(y_true, prediction, beta=THRESHOLD_BETA, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, prediction)),
        "roc_auc": safe_probability_metric(roc_auc_score, y_true, probability),
        "average_precision": average_precision,
        "average_precision_lift": (float(average_precision_lift) if np.isfinite(average_precision_lift) else np.nan),
        "brier_score": float(brier_score_loss(y_true, probability)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }


# Ranks every row by predicted probability and checks what fraction of the
# actual crash-positive rows fall inside the top k% of that ranking. This is
# the "if we only had time to inspect the riskiest k% of locations, how much
# of the real problem would we catch" metric — more intuitive to explain than
# a probability threshold, and it mirrors how a resource-limited city would
# actually use these rankings.
def top_k_hotspot_recovery(y_true, probability, k_fraction: float) -> float:
    y_true = np.asarray(y_true)
    probability = np.asarray(probability)

    total_positive = int(y_true.sum())
    if total_positive == 0:
        return np.nan

    # Number of rows in the top k% of the ranking (at least 1 row).
    top_k_count = max(1, int(np.ceil(k_fraction * len(probability))))

    # Indices of the top_k_count highest predicted-probability rows.
    top_k_indices = np.argsort(probability)[::-1][:top_k_count]

    positives_in_top_k = int(y_true[top_k_indices].sum())

    return positives_in_top_k / total_positive


# Finds the probability threshold that maximizes the F-beta score (beta from THRESHOLD_BETA,
# favoring recall). Uses ONE precision-recall curve, which evaluates every distinct predicted
# score in O(n log n). The earlier version called scikit-learn once per candidate threshold,
# which is fine for ~50k rows but would run for days on millions of validation rows.
# Returns the best threshold plus a table of precision/recall/F-beta across thresholds
# (thinned to at most `max_table_rows` rows, for plotting).
def find_best_fbeta_threshold(y_true, probability, beta: float = 2.0, max_table_rows: int = 3000) -> tuple[float, pd.DataFrame]:
    y_true = np.asarray(y_true)
    probability = np.asarray(probability, dtype=float)

    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    # precision/recall have one more entry than thresholds (the "no predictions" endpoint); drop it.
    precision, recall = precision[:-1], recall[:-1]

    in_range = (thresholds >= 0.0) & (thresholds <= 1.0)
    thresholds, precision, recall = thresholds[in_range], precision[in_range], recall[in_range]

    beta_squared = beta**2
    denominator = beta_squared * precision + recall
    fbeta = np.divide((1 + beta_squared) * precision * recall, denominator, out=np.zeros_like(denominator), where=denominator > 0)

    # Best F-beta, breaking ties by higher recall then higher precision (last key is primary).
    best_index = int(np.lexsort((precision, recall, fbeta))[-1])

    keep = np.unique(np.concatenate([np.linspace(0, len(thresholds) - 1, min(max_table_rows, len(thresholds))).astype(int), [best_index]]))
    threshold_table = pd.DataFrame({"threshold": thresholds[keep], "precision": precision[keep], "recall": recall[keep], "fbeta": fbeta[keep]})

    return (float(thresholds[best_index]), threshold_table)


# 25. Probability calibration

Calibration compares predicted probabilities with observed outcome frequencies.

A model can rank locations reasonably well while still producing probabilities that are too high or too low. If calibration is weak, the output should be described as a relative risk score rather than a literal probability.

In [ ]:
# Don't ask for more calibration bins than there are positive test examples —
# with too few positives, some bins would end up empty or unstable.
number_of_calibration_bins = min(10, max(2, int(y_test.sum())))

(
    # Group test rows into bins of similar predicted probability (quantile-based,
    # so each bin has roughly equal size), then compare each bin's average predicted
    # probability to its actual observed positive rate.
    observed_positive_fraction,
    mean_predicted_probability,
) = calibration_curve(y_test, test_probability, n_bins=number_of_calibration_bins, strategy="quantile")

# Turn the calibration-curve outputs into a readable table.
calibration_table = pd.DataFrame({"mean_predicted_probability": (mean_predicted_probability), "observed_positive_fraction": (observed_positive_fraction)})

# Positive gap = model underestimates risk in that bin; negative = overestimates.
calibration_table["calibration_gap"] = calibration_table["observed_positive_fraction"] - calibration_table["mean_predicted_probability"]

display(calibration_table)

# Plot observed vs. predicted probability per bin against the diagonal line
# that represents perfect calibration.
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(mean_predicted_probability, observed_positive_fraction, marker="o", label="Selected model")

ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

ax.set_title("Test probability calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed positive fraction")
ax.legend()

plt.tight_layout()
plt.show()

# 26. Permutation importance

Permutation importance measures how much test Average Precision decreases when one input column is shuffled.

It describes predictive dependence, not causation. Correlated variables can share or mask importance.

In [ ]:
# Permutation importance re-runs the model many times, so on a large test set
# it's capped to a random sample of rows to keep runtime reasonable.
if len(X_test) > PERMUTATION_SAMPLE_ROWS:
    importance_sample = test_data.sample(PERMUTATION_SAMPLE_ROWS, random_state=RANDOM_SEED)
else:
    importance_sample = test_data.copy()

X_importance = importance_sample[ALL_FEATURES]

y_importance = importance_sample[TARGET_COLUMN]

# For each feature, shuffle its values (breaking its relationship with the
# target) and measure how much the model's Average Precision drops — a bigger
# drop means the model relies on that feature more.
importance_result = permutation_importance(selected_model, X_importance, y_importance, scoring="average_precision", n_repeats=PERMUTATION_REPEATS, random_state=RANDOM_SEED, n_jobs=-1)

# Collect each feature's average importance (and its variability across
# repeats), sorted from most to least important.
importance_table = pd.DataFrame({"feature": ALL_FEATURES, "importance_mean": (importance_result.importances_mean), "importance_std": (importance_result.importances_std)}).sort_values(
    "importance_mean", ascending=False
)

display(importance_table)

# Plot only the top 15 most important features for readability.
top_importance = importance_table.head(15).sort_values("importance_mean")

ax = top_importance.plot(x="feature", y="importance_mean", kind="barh", figsize=(10, 7), legend=False, title=("Permutation importance on test data"))

ax.set_xlabel("Decrease in Average Precision after shuffling")

plt.tight_layout()
plt.show()

# 27. Test error analysis

False positives and false negatives should be examined separately, and neither should be over-interpreted.

Many false positives have more recent crash activity than true negatives do, so they deserve follow-up investigation rather than being dismissed. The model cannot prove that those intersections are dangerous, only that they resemble intersections that had a crash the following week. A false negative represents a positive intersection-week that the model failed to flag.

In [ ]:
# Columns to carry into the error-analysis table alongside predictions.
error_columns = ["node_id", "week_start", "latitude", "longitude", "street_count", "node_highway", "crashes_last_1w", "crashes_last_4w", "crashes_last_8w", "weeks_since_last_crash", TARGET_COLUMN]

# Start from the test split and attach each row's predicted probability/class.
error_table = test_data[error_columns].copy()

error_table["predicted_probability"] = test_probability

error_table["predicted_class"] = test_prediction

# Define the four possible outcomes by comparing actual target vs. predicted class.
error_conditions = [
    ((error_table[TARGET_COLUMN] == 0) & (error_table["predicted_class"] == 0)),
    ((error_table[TARGET_COLUMN] == 0) & (error_table["predicted_class"] == 1)),
    ((error_table[TARGET_COLUMN] == 1) & (error_table["predicted_class"] == 0)),
    ((error_table[TARGET_COLUMN] == 1) & (error_table["predicted_class"] == 1)),
]

error_labels = ["true_negative", "false_positive", "false_negative", "true_positive"]

# Label every row with which of the four outcomes it falls into.
error_table["error_type"] = np.select(error_conditions, error_labels, default="unknown")

display(error_table["error_type"].value_counts().to_frame("rows"))

In [ ]:
# For each outcome type, summarize row count and average feature values —
# useful for spotting patterns in what the model gets wrong.
error_summary = error_table.groupby("error_type", as_index=False).agg(
    rows=("node_id", "size"),
    mean_probability=("predicted_probability", "mean"),
    mean_street_count=("street_count", "mean"),
    mean_crashes_last_4w=("crashes_last_4w", "mean"),
    mean_crashes_last_8w=("crashes_last_8w", "mean"),
    mean_weeks_since_last_crash=("weeks_since_last_crash", "mean"),
)

display(error_summary)

# Missed crashes the model was MOST confident (lowest probability) would be safe —
# these are the riskiest kind of error to review first.
print("Most confident false negatives:")

display(error_table.loc[error_table["error_type"] == "false_negative"].sort_values("predicted_probability", ascending=True).head(25))

# False alarms the model was MOST confident about — worth checking whether
# they share some misleading pattern.
print("Most confident false positives:")

display(error_table.loc[error_table["error_type"] == "false_positive"].sort_values("predicted_probability", ascending=False).head(25))

## Presenting risk to the app

The eventual app should not state a raw probability (e.g. "68% chance of a crash") — calibration is currently weak enough that such a number would overstate precision. Instead, each intersection-week gets a **relative risk tier**, ranked against other predictions from the same period. Probabilities are still used internally for every evaluation metric in this notebook; only the presentation layer changes.

In [ ]:
# --- Relative risk tiers, not raw probabilities ---
# Calibration is currently weak (see section 25), so a number like "68%
# chance of a crash" would overstate how precise these predictions actually
# are. Instead, rank intersection-weeks against each other for the same
# period and report a relative risk TIER. Raw probabilities are still kept
# internally for every evaluation step above and below — only the
# app-facing presentation changes here.

def assign_risk_tier(probability: np.ndarray) -> pd.Series:
    # Percentile rank within this specific set of predictions (0-100, higher
    # = more relatively risky than other locations in the same period).
    percentile = pd.Series(probability).rank(pct=True) * 100

    tier_conditions = [
        percentile >= 90,
        percentile >= 75,
        percentile >= 50,
    ]
    tier_labels = ["Highest-priority (top 10%)", "Elevated relative risk", "Moderate relative risk"]

    return pd.Series(np.select(tier_conditions, tier_labels, default="Lower relative risk"), index=pd.Series(probability).index)


error_table["risk_percentile"] = pd.Series(error_table["predicted_probability"]).rank(pct=True) * 100

error_table["risk_tier"] = assign_risk_tier(error_table["predicted_probability"])

print("Risk tier distribution across the test period:")
display(error_table["risk_tier"].value_counts().to_frame("rows"))

# 28. Performance by subgroup

In [ ]:
# Computes the same classification_metrics(...) but restricted to one subset
# (subgroup) of rows, so performance can be compared across subgroups.
def subgroup_metrics(dataframe: pd.DataFrame, group_name: str) -> dict[str, Any]:
    probability = dataframe["predicted_probability"].to_numpy()

    target = dataframe[TARGET_COLUMN].to_numpy()

    metrics = classification_metrics(target, probability, selected_threshold)

    return {"group": group_name, "rows": len(dataframe), "positives": int(target.sum()), **metrics}


# Define subgroups of interest: simpler vs. more complex intersections, and
# intersections with vs. without recent crash history.
subgroup_definitions = {
    "street_count_3": (error_table["street_count"] == 3),
    "street_count_4_or_more": (error_table["street_count"] >= 4),
    "no_crash_last_4w": (error_table["crashes_last_4w"] == 0),
    "one_or_more_crashes_last_4w": (error_table["crashes_last_4w"] > 0),
}

# Compute metrics for every subgroup that actually has rows in the test data.
subgroup_rows = []

for group_name, mask in subgroup_definitions.items():
    group_data = error_table.loc[mask].copy()

    if len(group_data) == 0:
        continue

    subgroup_rows.append(subgroup_metrics(group_data, group_name))

subgroup_table = pd.DataFrame(subgroup_rows)

display(subgroup_table[["group", "rows", "positives", "positive_rate", "precision", "recall", "f2", "average_precision", "average_precision_lift", "brier_score"]])

# 29. Performance over the test period

In [ ]:
# Recompute classification metrics separately for each test week, to check
# whether performance is stable over time or drifts/degrades toward the end.
weekly_performance_rows = []

for week_start, week_data in error_table.groupby("week_start"):
    weekly_metrics = classification_metrics(week_data[TARGET_COLUMN], week_data["predicted_probability"], selected_threshold)

    weekly_performance_rows.append({"week_start": week_start, "rows": len(week_data), "positives": int(week_data[TARGET_COLUMN].sum()), **weekly_metrics})

# Assemble the per-week metrics into one table, ordered chronologically.
weekly_performance = pd.DataFrame(weekly_performance_rows).sort_values("week_start")

display(weekly_performance[["week_start", "positives", "positive_rate", "precision", "recall", "average_precision", "average_precision_lift", "brier_score"]])

# Plot Average Precision week by week across the test period.
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(weekly_performance["week_start"], weekly_performance["average_precision"], marker="o")

ax.set_title("Test Average Precision by week")
ax.set_xlabel("Week")
ax.set_ylabel("Average Precision")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 30. Map the highest-ranked test intersections

The map keeps only the highest-scoring test week for each node so that the same intersection is not plotted repeatedly.

A marker is a model output, not a permanent engineering judgment.

In [ ]:
# Cap how many intersections get plotted on the map, to keep it readable.
TOP_MAP_INTERSECTIONS = 100

# Take each intersection's single highest predicted-risk week, then keep only
# the highest-ranked N intersections overall.
top_ranked_intersections = error_table.sort_values("predicted_probability", ascending=False).drop_duplicates(subset=["node_id"], keep="first").head(TOP_MAP_INTERSECTIONS).copy()

# Re-rank tiers within just this displayed subset, so the map's tiers reflect
# relative risk among the locations actually being shown, not the full test set.
top_ranked_intersections["risk_tier"] = assign_risk_tier(top_ranked_intersections["predicted_probability"])

# Center the map on the median location of the plotted intersections.
map_center = [float(top_ranked_intersections["latitude"].median()), float(top_ranked_intersections["longitude"].median())]

# Create an interactive web map (folium wraps the Leaflet.js JS library).
risk_map = folium.Map(location=map_center, zoom_start=11)

# Add one marker per intersection, with a popup showing its key details and
# a tooltip showing its risk score at a glance.
for _, row in top_ranked_intersections.iterrows():
    popup_text = (
        f"Node: {row['node_id']}<br>"
        f"Week: {row['week_start'].date()}<br>"
        f"Relative risk: "
        f"{row['risk_tier']}<br>"
        f"Actual next-week target: "
        f"{int(row[TARGET_COLUMN])}<br>"
        f"Previous 4-week crashes: "
        f"{row['crashes_last_4w']}<br>"
        f"Street count: "
        f"{row['street_count']}<br>"
        f"Outcome: {row['error_type']}"
    )

    # The raw probability still exists internally (row["predicted_probability"])
    # but is not shown here — only the relative risk tier is app-facing.
    folium.Marker(location=[row["latitude"], row["longitude"]], popup=folium.Popup(popup_text, max_width=350), tooltip=row["risk_tier"]).add_to(risk_map)

# Save the interactive map as a standalone HTML file.
MAP_OUTPUT_PATH = NOTEBOOK_OUTPUT_DIR / "safecross_top_ranked_test_map.html"

risk_map.save(MAP_OUTPUT_PATH)

print("Saved map:", MAP_OUTPUT_PATH)
risk_map

# 31. Separate prediction evidence from intervention evidence

The model can identify predictive patterns. It cannot independently determine which road-design intervention should be implemented.

For each high-ranked location, Joey should combine model output with:

- Crash type and severity
- Traffic and pedestrian exposure
- Crossing distance and geometry
- Signal timing
- Speed environment
- Lighting and visibility
- Land use, transit, and school context
- Published evidence on intervention effectiveness

In [ ]:
# A reference table separating what the model's output can and can't support:
# each row pairs a model signal with what it does and doesn't prove, to guide
# how predictions should (and shouldn't) inform real interventions.
recommendation_review_template = pd.DataFrame(
    [
        {
            "model_signal": ("High recent crash history"),
            "what_the_model_supports": ("The location shows persistent " "reported-crash activity."),
            "what_requires_external_evidence": ("Which intervention addresses " "the relevant crash types."),
        },
        {
            "model_signal": ("High intersection complexity"),
            "what_the_model_supports": ("Complexity is associated with " "the model output."),
            "what_requires_external_evidence": ("Whether geometry caused the " "observed crashes."),
        },
        {
            "model_signal": ("Weather dependence"),
            "what_the_model_supports": ("The risk ranking changes with " "historical weather context."),
            "what_requires_external_evidence": ("Whether weather-responsive " "operations would reduce risk."),
        },
        {
            "model_signal": ("Coordinate dependence"),
            "what_the_model_supports": ("The model detects persistent " "spatial patterns."),
            "what_requires_external_evidence": ("Why the pattern exists and " "what should change."),
        },
    ]
)

display(recommendation_review_template)

# 31b. Build the app-ready per-intersection output table

This is the table Notebook 3's outputs are ultimately organized around, and what the Streamlit app will read directly. Each row is one selected intersection with: location, relative risk tier/percentile, recent crash history, key road features, its top 3 model signals, a short plain-language explanation, candidate safety measures for further review, and a limitation statement. Candidate measures are simple, transparent rules — not model output — and are explicitly framed as starting points for review, not recommendations.

In [ ]:
# Build one clean row per selected intersection, in a shape that can feed
# almost directly into the eventual Streamlit app. This is a first pass —
# it's meant to be refined further, not treated as a finished UI spec.

APP_TOP_N_INTERSECTIONS = 25

# "Top 3 model signals" here means: among the globally-important features (from the
# permutation importance table above), the 3 whose values are most unusually HIGH for this
# specific row, measured as the share of all test-period rows with a strictly lower value.
# Comparing percentiles rather than raw values matters because the features are on very
# different scales (a crash count of 2 vs. a latitude of 40.7). This is a simple, explainable
# approximation, not a per-row attribution method (e.g. SHAP): good enough to point a
# reviewer toward what likely drove a high ranking, not a proof of causation.
#
# Only NUMERIC features can be ranked this way. Text features (e.g. road class) are skipped
# here; road characteristics are still reported in "key_road_features" below.
candidate_signal_features = [
    feature
    for feature in importance_table["feature"].tolist()
    if feature in error_table.columns and pd.api.types.is_numeric_dtype(error_table[feature])
][:8]

# Sorted copies of each feature's test-period values, used to look up percentiles quickly.
sorted_signal_values = {
    feature: np.sort(error_table[feature].dropna().to_numpy(dtype=float))
    for feature in candidate_signal_features
}


def signal_percentile(feature: str, value: float) -> float:
    reference = sorted_signal_values[feature]
    return np.searchsorted(reference, value, side="left") / len(reference)


def top_signals_for_row(row) -> str:
    scored = [
        (feature, signal_percentile(feature, float(row[feature])), row[feature])
        for feature in candidate_signal_features
        if pd.notna(row[feature])
    ]
    ranked = sorted(scored, key=lambda item: item[1], reverse=True)[:3]
    return "; ".join(f"{feature}={float(value):.4g} (p{percentile * 100:.0f})" for feature, percentile, value in ranked)


# A simple, transparent rule-based mapping from top signals to CANDIDATE
# measures for further review — never a final recommendation. These exist to
# give a reviewer a starting point, not to replace site review or published
# safety evidence (see section 31 above).
def candidate_measures_for_row(row) -> str:
    measures = []

    if row.get("pedestrian_casualty_crashes_last_8w", 0) and row["pedestrian_casualty_crashes_last_8w"] > 0:
        measures.append("Review crosswalk marking/visibility")

    if row.get("cyclist_casualty_crashes_last_8w", 0) and row["cyclist_casualty_crashes_last_8w"] > 0:
        measures.append("Review bike-lane/intersection conflict design")

    if row.get("max_connecting_speed_limit") and pd.notna(row["max_connecting_speed_limit"]) and row["max_connecting_speed_limit"] >= 35:
        measures.append("Review traffic-calming / speed-limit appropriateness")

    if row.get("is_traffic_signal", 0) == 0 and row.get("street_count", 0) and row["street_count"] >= 4:
        measures.append("Review whether signal or stop control is warranted")

    if not measures:
        measures.append("No rule-based candidate measure triggered — recommend manual site review")

    return "; ".join(measures)


app_source_rows = top_ranked_intersections.head(APP_TOP_N_INTERSECTIONS).copy()

app_output_table = pd.DataFrame(
    {
        "intersection_node_id": app_source_rows["node_id"],
        "latitude": app_source_rows["latitude"],
        "longitude": app_source_rows["longitude"],
        "relative_risk_tier": app_source_rows["risk_tier"],
        "relative_risk_percentile": app_source_rows["predicted_probability"].rank(pct=True).mul(100).round(1),
        "recent_crash_history": (
            "Last 4 weeks: " + app_source_rows["crashes_last_4w"].astype("string") + " crash(es); weeks since last crash: " + app_source_rows["weeks_since_last_crash"].astype("string")
        ),
        "key_road_features": (
            "Street count: " + app_source_rows["street_count"].astype("string") + ", class: " + app_source_rows.get("node_highway", pd.Series(["unknown"] * len(app_source_rows))).astype("string")
        ),
        "top_3_model_signals": app_source_rows.apply(top_signals_for_row, axis=1),
        "short_explanation": (
            "Ranked " + app_source_rows["risk_tier"].astype("string") + " relative to other intersections in this period, based mainly on recent crash "
            "history and connecting road characteristics."
        ),
        "candidate_safety_measures": app_source_rows.apply(candidate_measures_for_row, axis=1),
        "limitation_caution": (
            "Model output is not exposure-normalized and does not prove causation. "
            "Candidate measures require site review and external road-safety evidence "
            "before any action is taken."
        ),
    }
)

display(app_output_table.head())

APP_OUTPUT_TABLE_PATH = NOTEBOOK_OUTPUT_DIR / "app_intersection_recommendations.csv"
app_output_table.to_csv(APP_OUTPUT_TABLE_PATH, index=False)
print("Saved app-ready output table:", APP_OUTPUT_TABLE_PATH)

# 32. Create the model card

In [ ]:
# A standard "model card": a structured summary of what this model is, what
# it's for, what it should NOT be used for, its data sources, known
# limitations, and its final test performance — intended for anyone reviewing
# or deciding whether to trust/use this model.
model_card = {
    "model_name": ("SafeCross AI intersection-week classifier"),
    "selected_model": selected_model_name,
    "prediction_unit": metadata["prediction_unit"],
    "prediction_horizon": metadata["prediction_horizon"],
    "target": TARGET_COLUMN,
    "intended_use": ("Research screening and prioritization " "for follow-up road-safety analysis."),
    "not_intended_for": ("Automatic engineering decisions, " "causal claims, enforcement decisions, " "or replacement of professional review."),
    "major_data_sources": ["NYC reported crash events", "OpenStreetMap network attributes", "Historical weather summaries"],
    "known_limitations": [
        ("Traffic and pedestrian exposure " "are not directly modeled."),
        ("Reported crashes may contain " "reporting and location error."),
        ("Nearest-node assignment is an " "approximation."),
        ("Current OSM tags may not represent " "historical road conditions."),
        ("Coordinates may support hotspot " "memorization."),
        ("One borough-level weather series " "cannot explain spatial variation " "within a week."),
        ("One year provides limited temporal " "validation."),
    ],
    "test_metrics": test_metrics,
}

# Save the model card to disk as JSON.
MODEL_CARD_PATH = NOTEBOOK_OUTPUT_DIR / "safecross_model_card.json"

with open(MODEL_CARD_PATH, "w", encoding="utf-8") as file:
    json.dump(model_card, file, indent=2, allow_nan=True)

print("Saved model card:", MODEL_CARD_PATH)
display(pd.Series(model_card, name="value").to_frame())

# 33. Save explainability and error-analysis outputs

In [ ]:
# File paths for every explainability/error-analysis table produced above.
CALIBRATION_PATH = NOTEBOOK_OUTPUT_DIR / "calibration_table.csv"

IMPORTANCE_PATH = NOTEBOOK_OUTPUT_DIR / "permutation_importance.csv"

ERROR_ANALYSIS_PATH = NOTEBOOK_OUTPUT_DIR / "test_error_analysis.parquet"

SUBGROUP_PATH = NOTEBOOK_OUTPUT_DIR / "subgroup_performance.csv"

WEEKLY_PERFORMANCE_PATH = NOTEBOOK_OUTPUT_DIR / "weekly_test_performance.csv"

RECOMMENDATION_TEMPLATE_PATH = NOTEBOOK_OUTPUT_DIR / "recommendation_review_template.csv"

# Write each table out to its corresponding file.
calibration_table.to_csv(CALIBRATION_PATH, index=False)

importance_table.to_csv(IMPORTANCE_PATH, index=False)

error_table.to_parquet(ERROR_ANALYSIS_PATH, index=False)

subgroup_table.to_csv(SUBGROUP_PATH, index=False)

weekly_performance.to_csv(WEEKLY_PERFORMANCE_PATH, index=False)

recommendation_review_template.to_csv(RECOMMENDATION_TEMPLATE_PATH, index=False)

print("Saved analysis outputs to:")
print(NOTEBOOK_OUTPUT_DIR)

> **Note:** the numbers below come from the earlier Manhattan-only, one-year run. After re-running Notebooks 1 and 2 on the full NYC history, every figure here (winning model, average precision, ablation effects, calibration) must be regenerated. Do not reuse them.

1. Modeling weeks: 51 weeks of complete intersection-week data went into modeling (train ~35 weeks, validation 8, test 8), and the positive class (a crash the following week) makes up roughly 4.6–4.9% of rows across splits.

2. Positive-class prevalence: About 4.05% of training rows, 4.65% of validation rows, and 4.94% of test rows are positive — confirming this is a rare-event problem.

3. Winning model: Logistic regression was selected over random forest, edging it out on the primary metric (validation average precision 0.0944 vs. 0.0900), even though the two performed almost identically overall.

4. Beating the baseline: Yes — logistic regression's validation average-precision lift (2.03x random chance) clearly beat the historical-4-week heuristic's lift (1.28x), so learning from features paid off over a simple "recently active = risky" rule.

5. Selected threshold: The validation-tuned decision threshold was 0.4855, chosen to optimize F2 (favoring recall over precision).

6. Effect of removing coordinates: Dropping latitude/longitude barely hurt performance (average precision 0.0942 vs. 0.0944 full model), so the model isn't leaning heavily on raw location as a hotspot shortcut.

7. Effect of removing crash history: Removing crash-history features caused the biggest drop of any ablation (average precision fell from 0.0944 to 0.0694), showing the model depends heavily on an intersection's recent track record.

8. Did weather help? No — removing weather features actually left average precision essentially unchanged (0.0960 vs. 0.0944), so weather added little to no real predictive value.

9. What is the model mainly learning? Given ablations 6–8, the model is primarily learning short-term temporal persistence ("this place has been active lately"), not spatial hotspot identity or weather effects.

10. Overly strong conclusion to avoid: It would be too strong to say the model identifies "dangerous intersections" in a causal sense — it's really just detecting recent-activity persistence, so a high score reflects a statistical pattern, not a verified engineering risk factor.

11. Final test performance: On the untouched test period the locked model scored precision 0.095, recall 0.508, F2 0.271, ROC AUC 0.698, and average precision 0.102 (a 2.06x lift over chance) — close to its validation numbers, meaning it generalized reasonably well without overfitting.

12. Calibration quality: Calibration is weak and consistently overconfident — at every predicted-probability bucket the observed crash rate is much lower than predicted (e.g., predicted ~68% vs. observed ~11% in the top bucket), so outputs should be read as relative risk rankings, not literal probabilities.

13. Top predictive features: Permutation importance ranks `weeks_since_last_crash` and `crashes_last_8w` far above everything else, reinforcing that recency/frequency of past crashes drives the model, while individual weather variables and cyclist/pedestrian-specific counts matter little.

14. False negatives vs. false positives: False negatives tend to be quieter, longer-dormant intersections (avg. 24.8 weeks since last crash) that the model reasonably judged low-risk, while false positives tend to be genuinely active locations (avg. 3.9 weeks since last crash, mean probability ~0.61) that just didn't happen to have a crash in that particular week — meaning many of these false positives deserve follow-up investigation rather than being dismissed — the model cannot prove these intersections are dangerous, only that they resemble ones that had a crash the following week.

15. Subgroup/time stability: Performance is uneven by subgroup — intersections with recent crash activity are predicted far better (average-precision lift 1.46–1.71x) than those without (lift 1.89–2.57x driven mostly by rarity, with much lower recall), and performance across the 8 test weeks is fairly stable (average precision roughly 0.08–0.14) without a clear trend, suggesting no major time-based drift within this window.